### LIBRARY IMPORTS

In [3]:
import yaml
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import log_loss
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_loader import DataLoader
from src.nn_regressor import NNRegressor

### CONFIGURATION

In [4]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

active_dataset = "heart"
active_dataset_config = config["datasets"][active_dataset]

data_loader = DataLoader(active_dataset, active_dataset_config["id_column"], active_dataset_config["target"])
data, _ = data_loader.load_processed_data()

X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns=[active_dataset_config["target"]]),
    data[active_dataset_config["target"]],
    test_size = 0.8,
    random_state = 42
)

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

### LOGISTIC REGRESSION

In [35]:
logistic = LogisticRegressionCV(Cs=10, cv=5)
logistic.fit(X_train, y_train)

logistic_test_preds = logistic.predict_proba(X_test)

print('-' * 50)
print(f"Logistic best hyperparameter: {logistic.C_[-1]}")
print(f"Logistic validation cross entropy: {log_loss(y_test, logistic_test_preds)}")

c:\Projects\TFG\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Projects\TFG\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


--------------------------------------------------
Logistic best hyperparameter: 0.046415888336127774
Logistic validation cross entropy: 0.27393431549010194


### SUPPORT VECTOR MACHINE

In [6]:
linear_svm = SVC(kernel='linear', C=1.0, probability=True)
linear_svm.fit(X_train, y_train)

linear_svm_preds = linear_svm.predict_proba(X_test)

rbf_svm = SVC(kernel='rbf', gamma='scale', C=1.0, probability=True)
rbf_svm.fit(X_train, y_train)

rbf_svm_preds = rbf_svm.predict_proba(X_test)

print(f"Linear SVM cross entropy: {log_loss(y_test, linear_svm_preds)}")
print(f"RBF SVM cross entropy: {log_loss(y_test, rbf_svm_preds)}")

Linear SVM cross entropy: 0.27422544802725946
RBF SVM cross entropy: 0.29541181470610334


### RANDOM FOREST

In [36]:
rf = RandomForestClassifier(n_estimators=1_000, max_depth=5, max_features="sqrt")
rf.fit(X_train, y_train)

rf_test_preds = rf.predict_proba(X_test)

print(f"Random forest validation cross entropy: {log_loss(y_test, rf_test_preds)}")

Random forest validation cross entropy: 0.3188188598584369


### NEURAL NETWORK

In [37]:
class MLPClassifier(NNRegressor):
    def __init__(self, hidden_size=64, batch_size=64, dropout=0.3):
        super().__init__(hidden_size, batch_size)
        self.dropout = dropout

    def fit(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_valid: np.ndarray,
        y_valid: np.ndarray, 
        epochs: int = 100,
        patience: int = 20,
        lr: float = 1e-3
    ) -> None:
        input_size = X_train.shape[1]
        
        unique_classes = np.unique(y_train)
        output_size = len(unique_classes)
        
        self._get_network(input_size, output_size)
        
        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).to(torch.long).to(self.device).squeeze()

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters(), lr=lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(epochs):
            self.train()
            train_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                loss = criterion(preds, batch_y.to(torch.long).to(self.device).squeeze())
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()
                scheduler.step(val_loss)

            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch + 1}. Validation cross entropy: {val_loss:.6f}. Learning rate: {current_lr}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                print(f"Early stopping on epoch {epoch + 1}!! Best validation MSE: {best_loss:.6f}")
                break
                
        if best_model:
            self.load_state_dict(best_model)

    def _get_network(self, input_size: int, output_size: int) -> None:
        self.network = nn.Sequential(
            nn.Linear(input_size, self.hidden_size),
            nn.BatchNorm1d(self.hidden_size),
            nn.GELU(),
            nn.Dropout(self.dropout),

            nn.Linear(self.hidden_size, self.hidden_size),
            nn.BatchNorm1d(self.hidden_size),
            nn.GELU(),
            nn.Dropout(self.dropout),

            nn.Linear(self.hidden_size, output_size)
        )

        self.network.to(self.device)

mlp_regressor = MLPClassifier()
mlp_regressor.fit(X_train.values, y_train, X_test.values, y_test)

Epoch 1. Validation cross entropy: 0.275564. Learning rate: 0.001
Epoch 2. Validation cross entropy: 0.276482. Learning rate: 0.001
Epoch 3. Validation cross entropy: 0.274659. Learning rate: 0.001
Epoch 4. Validation cross entropy: 0.275581. Learning rate: 0.001
Epoch 5. Validation cross entropy: 0.274218. Learning rate: 0.001
Epoch 6. Validation cross entropy: 0.275069. Learning rate: 0.001
Epoch 7. Validation cross entropy: 0.274650. Learning rate: 0.001
Epoch 8. Validation cross entropy: 0.274860. Learning rate: 0.001
Epoch 9. Validation cross entropy: 0.274667. Learning rate: 0.001
Epoch 10. Validation cross entropy: 0.275175. Learning rate: 0.001
Epoch 11. Validation cross entropy: 0.274740. Learning rate: 0.0005
Epoch 12. Validation cross entropy: 0.274576. Learning rate: 0.0005
Epoch 13. Validation cross entropy: 0.274465. Learning rate: 0.0005
Epoch 14. Validation cross entropy: 0.275885. Learning rate: 0.0005
Epoch 15. Validation cross entropy: 0.274740. Learning rate: 0.0005